# 07 — Validación cruzada y búsqueda de hiperparámetros

Objetivos:
- Entender por qué un solo `train_test_split` puede ser inestable.
- Usar `StratifiedKFold` para clasificación y `KFold` para regresión.
- Comparar modelos con `cross_validate`.
- Ajustar hiperparámetros con `GridSearchCV` y `RandomizedSearchCV`.
- Conectar todo con `sklearn.pipeline.Pipeline`: preprocesamiento + modelo dentro de una sola unidad reproducible.

In [ ]:
%load_ext kedro.ipython

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "raw" / "database.sqlite"
assert (PROJECT_ROOT / "pyproject.toml").is_file(), (
    "Abre este notebook desde la raíz del proyecto con `kedro jupyter lab`."
)
assert DB_PATH.is_file(), (
    "No existe data/raw/database.sqlite. Ejecuta: `python scripts/bootstrap_data.py`."
)
print(f"Proyecto: {PROJECT_ROOT}")
print(f"SQLite: {DB_PATH}")

## 1. Dataset de modelado

Usaremos las mismas columnas que el pipeline Kedro: cuotas Bet365 (`B365H`, `B365D`, `B365A`) y targets derivados desde goles.

In [ ]:
import numpy as np
import pandas as pd

match = catalog.load("match")
goal_diff = match["home_team_goal"] - match["away_team_goal"]
outcome = np.select([goal_diff > 0, goal_diff == 0, goal_diff < 0], [2, 1, 0])
feat_cols = ["B365H", "B365D", "B365A"]
df = match[feat_cols].assign(
    outcome=outcome,
    home_team_goal=match["home_team_goal"],
    away_team_goal=match["away_team_goal"],
).dropna()

X = df[feat_cols]
y_clf = df["outcome"]
y_reg = df["home_team_goal"]
print(df.shape)
df.head()

## 2. Clasificación con StratifiedKFold

`StratifiedKFold` conserva proporciones de clases en cada fold. Para clasificación multiclase suele ser mejor que `KFold` simple cuando hay desbalance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

clf_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf_models = {
    "LogisticRegression": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2_000)),
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=150, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=1
    ),
}

rows = []
for name, model in clf_models.items():
    scores = cross_validate(
        model,
        X,
        y_clf,
        cv=clf_cv,
        scoring={"f1_macro": make_scorer(f1_score, average="macro"), "accuracy": "accuracy"},
        n_jobs=1,
    )
    rows.append({
        "modelo": name,
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std(),
        "accuracy_mean": scores["test_accuracy"].mean(),
    })

pd.DataFrame(rows).sort_values("f1_macro_mean", ascending=False)

## 3. GridSearchCV para clasificación

`GridSearchCV` prueba todas las combinaciones. Es claro para enseñar, pero puede crecer muy rápido.

In [ ]:
from sklearn.model_selection import GridSearchCV

log_grid = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2_000)),
])
param_grid = {
    "clf__C": [0.1, 1.0, 10.0],
    "clf__solver": ["lbfgs"],
}

grid = GridSearchCV(
    log_grid,
    param_grid=param_grid,
    cv=clf_cv,
    scoring="f1_macro",
    n_jobs=1,
)
grid.fit(X, y_clf)
print("Mejores hiperparámetros:", grid.best_params_)
print("Mejor F1 macro CV:", round(grid.best_score_, 4))

## 4. Regresión con KFold

Para targets continuos usamos `KFold`. Aquí comparamos MAE y RMSE; scikit-learn usa métricas negativas para errores porque internamente maximiza el score.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline as SkPipeline

reg_cv = KFold(n_splits=5, shuffle=True, random_state=42)
reg_models = {
    "Ridge": SkPipeline([("scale", StandardScaler()), ("reg", Ridge())]),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=150, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=1
    ),
}

rows = []
for name, model in reg_models.items():
    scores = cross_validate(
        model,
        X,
        y_reg,
        cv=reg_cv,
        scoring={"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"},
        n_jobs=1,
    )
    rows.append({
        "modelo": name,
        "mae_mean": -scores["test_mae"].mean(),
        "rmse_mean": -scores["test_rmse"].mean(),
        "r2_mean": scores["test_r2"].mean(),
    })

pd.DataFrame(rows).sort_values("mae_mean")

## 5. RandomizedSearchCV para regresión

`RandomizedSearchCV` explora una muestra de combinaciones. Es útil cuando el espacio de búsqueda es grande.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestRegressor(random_state=42, n_jobs=1)
param_distributions = {
    "n_estimators": [80, 120, 180],
    "max_depth": [6, 10, 16, None],
    "min_samples_leaf": [2, 5, 10],
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_distributions,
    n_iter=6,
    cv=reg_cv,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=1,
)
search.fit(X, y_reg)
print("Mejores hiperparámetros:", search.best_params_)
print("Mejor MAE CV:", round(-search.best_score_, 4))

## Cierre

Para presentar resultados honestos:
- Separar selección de hiperparámetros y evaluación final.
- Reportar media y desviación estándar por fold.
- Usar la métrica alineada con el problema: `f1_macro` si importan clases minoritarias; `MAE` si queremos error interpretable en goles.